### Data loader

In [5]:
import numpy as np
import pandas as pd
import pickle
import dill
import networkx as nx
from pathlib import Path
from scipy.sparse import load_npz
from tqdm.auto import tqdm

In [6]:
# Working directory
BASE_DIR = Path('/Users/gre_en/Documents/Analysis/Projects/1_research/Sci-Soc')

# Processed data directory
NETWORKS_DIR = BASE_DIR / "data" / "processed" / "networks"    # Network adj matrix
EMB_DIR = BASE_DIR / "data" / "processed" / "embeddings"       # Embedding vector directory
DYSAT_DIR = BASE_DIR / "data" / "processed" / "dysat_input"    # Input for the official DySAT code

# Result directory
PLOT_DIR = BASE_DIR / "results" / "figures"

DYSAT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START = 1990
YEAR_END   = 2023
years = list(range(YEAR_START, YEAR_END + 1))

RANDOM_SEED = 42
SOURCES = ["news", "paper"]

In [8]:
def load_node_info(source, net_dir):
    with open(Path(net_dir) / f"{source}_node_info.pkl", "rb") as f:
        return pickle.load(f)

info = {"news": load_node_info("news", NETWORKS_DIR),
        "paper": load_node_info("paper", NETWORKS_DIR)}

# sanity check
print(len(list(NETWORKS_DIR.glob("*_fixed_adj_*.npz"))), "matrices")
print(info["news"].keys())
print("news  years", len(info["news"]["years"]), "V_t", info["news"]["V_t"][0], "->", info["news"]["V_t"][-1])
print("paper years", len(info["paper"]["years"]), "V_t", info["paper"]["V_t"][0], "->", info["paper"]["V_t"][-1])

68 matrices
dict_keys(['vocab', 'years', 'concept_freq_year', 'concept_freq_total', 'concept_share', 'concept_share_all', 'n_docs_all', 'n_docs_ge2', 'V_t', 'E_t'])
news  years 34 V_t 2434 -> 7883
paper years 34 V_t 12593 -> 18159


In [9]:
import shutil

def to_dysat_input(source, info, net_dir, out_dir):
    out_dir = Path(out_dir) / source
    out_dir.mkdir(parents=True, exist_ok=True)
    for year in info["years"]:
        shutil.copy(Path(net_dir) / f"{source}_fixed_adj_{year}.npz",
                    out_dir / f"adj_{year}.npz")
    np.save(out_dir / "years.npy", np.asarray(info["years"]))
    np.save(out_dir / "vocab.npy", np.array(info["vocab"], dtype=object))
    np.save(out_dir / "active.npy", np.asarray(info["concept_freq_year"]) > 0)

for s in SOURCES:
    to_dysat_input(s, info[s], NETWORKS_DIR, DYSAT_DIR)

In [10]:
for s in SOURCES:
    d = DYSAT_DIR / s
    files = sorted(d.glob("adj_*.npz"))
    print(s, len(files), "files",
          f"{sum(f.stat().st_size for f in d.iterdir())/1e6:.1f} MB")
    if files:
        print("  ", files[-1].name, f"{files[-1].stat().st_size/1e6:.2f} MB")

news 34 files 557.6 MB
   adj_2023.npz 3.58 MB
paper 34 files 832.6 MB
   adj_2023.npz 15.82 MB
